## 1. Setup — Install & Imports

In [1]:
!pip install transformers -q

import pandas as pd
from transformers import pipeline
from tqdm.auto import tqdm
import json
import time


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


/home/lenovo/Documents/School/Sem6/PBA/pba-task-group/.env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load NER Model

In [2]:
MODEL_NAME = "cahya/bert-base-indonesian-NER"

print(f"Loading model: {MODEL_NAME} ...")
ner = pipeline(
    "ner",
    model=MODEL_NAME,
    aggregation_strategy="first",
)

Loading model: cahya/bert-base-indonesian-NER ...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 31665.33it/s]
[transformers] BertForTokenClassification LOAD REPORT from: cahya/bert-base-indonesian-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 3. Load Dataset

In [ ]:
CSV_PATH = "../preprocessed/dataset_preprocessed.csv"  # update path if running locally

df = pd.read_csv(CSV_PATH)

# Use the original 'konten' column for NER (richer text vs. preprocessed)
TEXT_COL = "konten_preprocessed"

# Drop rows with missing article text
df = df.dropna(subset=[TEXT_COL]).reset_index(drop=True)

print(f"Loaded {len(df):,} articles")
print(f"Columns: {df.columns.tolist()}")
df.head(3)

Loaded 999 articles
Columns: ['no', 'url', 'judul', 'konten', 'manual sentiment', 'konten_length', 'konten_preprocessed']


,no,url,judul,konten,manual sentiment,konten_length,konten_preprocessed
0,1,https://www.cnbcindonesia.com/research/2025012...,Trump Sebar Exceutive Order: Emang Semengerika...,"Jakarta, CNBC Indonesia -Amerika Serikat (AS) ...",Positive,8326,indonesia amerika serikat resmi milik presiden...
1,2,https://www.cnbcindonesia.com/research/2025012...,"Alasan Rupiah 'Berpesta' di Pelantikan Trump, ...","Jakarta, CNBC Indonesia -Nilai tukar rupiah te...",Positive,2925,indonesia nilai tukar rupiah dolar amerika ser...
2,3,https://www.cnbcindonesia.com/research/2025012...,"Trump Beri Kabar Baik, Saatnya Menunggu Dolar ...","Jakarta, CNBC Indonesia-Pasar keuangan Indones...",Positive,13102,indonesia pasar uang indonesia kuat lantik don...


## 4. Helper — Clean & Run NER

In [4]:
# Characters that should not appear as standalone entities
SKIP_TOKENS = {".", ",", "%", "(", ")", "-", "/", "'", '"'}

# Maximum characters to feed into the model per article
# NusaBERT has a 512-token limit; ~1500 chars is a safe approximate ceiling
MAX_CHARS = 1500


def clean_entities(raw_entities: list) -> list:
    """
    Filter and clean raw NER output.

    Returns a list of dicts with keys:
        word, entity_group, score, start, end
    """
    cleaned = []
    seen = set()  # deduplicate (word, label) pairs

    for ent in raw_entities:
        word = ent["word"].strip().strip(".,")

        # Skip empty or punctuation-only tokens
        if not word or word in SKIP_TOKENS:
            continue

        # Skip very short tokens (single characters) that are rarely meaningful
        if len(word) <= 1:
            continue

        label = ent["entity_group"]
        key = (word.lower(), label)

        cleaned.append({
            "word": word,
            "entity_group": label,
            "score": round(float(ent["score"]), 4),
            "start": ent.get("start"),
            "end": ent.get("end"),
            "is_duplicate": key in seen,
        })
        seen.add(key)

    return cleaned


def run_ner(text: str) -> list:
    """
    Truncate text to MAX_CHARS, run NER, return cleaned entities.
    Returns an empty list on error.
    """
    if not isinstance(text, str) or not text.strip():
        return []
    try:
        truncated = text[:MAX_CHARS]
        raw = ner(truncated)
        return clean_entities(raw)
    except Exception as exc:
        print(f"  ⚠️  NER error: {exc}")
        return []


def display_entities(entities: list, title: str = ""):
    """Pretty-print a list of entity dicts."""
    if title:
        print(f"\n{'='*60}")
        print(f"  {title}")
        print(f"{'='*60}")

    unique = [e for e in entities if not e["is_duplicate"]]
    print(f"{'Word':<35} | {'Label':<8} | {'Score'}")
    print("-" * 55)
    for ent in unique:
        print(f"{ent['word']:<35} | {ent['entity_group']:<8} | {ent['score']:.4f}")
    print(f"\n  → {len(unique)} unique entities found")

## 5. Single-Article Test (Sanity Check)

Run NER on article index `0` to verify everything works before processing the full dataset.

In [5]:
TEST_IDX = 0  # ← change to any row index to inspect a different article

test_row = df.iloc[TEST_IDX]
test_text = test_row[TEXT_COL]

print(f"Article index : {TEST_IDX}")
print(f"Title         : {test_row.get('judul', 'N/A')}")
print(f"Text length   : {len(test_text):,} chars (truncated to {MAX_CHARS} for NER)")
print(f"\nText preview  :\n{test_text[:400]}\n{'...' if len(test_text) > 400 else ''}")

test_entities = run_ner(test_text)
display_entities(test_entities, title=f"NER Results — Article {TEST_IDX}")

Article index : 0
Title         : Trump Sebar Exceutive Order: Emang Semengerikan Apa Buat Indonesia? - CNBC Indonesia
Text length   : 4,266 chars (truncated to 1500 for NER)

Text preview  :
indonesia amerika serikat resmi milik presiden sambut positif agenda pro bisnis bawa presiden amerika serikat donald trump resmi lantik senin pidato trump bijak dagang proteksionis khawatir agresif terap ukur trump bawa agenda ambisius liput reformasi dagang imigrasi mangkas pajak deregulasi bijak potensi tingkat untung korporasi risiko picu inflasi tekan saham obligasi dorong bank sentral fed nai
...

  NER Results — Article 0
Word                                | Label    | Score
-------------------------------------------------------
indonesia                           | GPE      | 0.9353
amerika serikat                     | GPE      | 0.7531
presiden                            | NOR      | 0.4955
presiden amerika serikat            | NOR      | 0.9042
donald trump                        | PER

## 6. Full Dataset NER Run

Iterate over every article, collect entities, and build a results DataFrame.


In [6]:
all_results = []
error_indices = []

start_time = time.time()

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Running NER"):
    text = row[TEXT_COL]
    entities = run_ner(text)

    if not entities:
        error_indices.append(idx)

    for ent in entities:
        all_results.append({
            "article_idx"  : idx,
            "judul"        : row.get("judul"),
            "url"          : row.get("url"),
            "word"         : ent["word"],
            "entity_group" : ent["entity_group"],
            "score"        : ent["score"],
            "is_duplicate" : ent["is_duplicate"],
            "start"        : ent["start"],
            "end"          : ent["end"],
        })

elapsed = time.time() - start_time
print(f"\n Done in {elapsed:.1f}s")
print(f"   Articles processed       : {len(df):,}")
print(f"   Total entities           : {len(all_results):,}")
print(f"   Articles with no entities: {len(error_indices)}")
if error_indices:
    print(f"   Failed indices: {error_indices[:10]}{'...' if len(error_indices) > 10 else ''}")

Running NER: 100%|██████████| 999/999 [00:25<00:00, 38.58it/s]


 Done in 25.9s
   Articles processed       : 999
   Total entities           : 38,064
   Articles with no entities: 0


## 7. Explore Results

In [7]:
results_df = pd.DataFrame(all_results)

print(f"Results shape: {results_df.shape}")
print(f"\nEntity label distribution:")
print(results_df["entity_group"].value_counts())

print(f"\nTop 20 most frequent unique entities (all labels):")
top_entities = (
    results_df
    .groupby(["word", "entity_group"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(20)
)
print(top_entities.to_string(index=False))

Results shape: (38064, 9)

Entity label distribution:
entity_group
GPE    9882
ORG    8026
PRD    6145
PER    5204
NOR    3352
DAT    1782
EVT    1294
MON    1239
LAW     290
PRC     283
LOC     257
CRD     216
TIM      41
ORD      25
QTY      19
REG       5
FAC       2
WOA       2
Name: count, dtype: int64

Top 20 most frequent unique entities (all labels):
                    word entity_group  count
                   trump          PER   2236
               indonesia          GPE   1837
                   china          GPE   1826
            donald trump          PER    784
                 amerika          GPE    676
         amerika serikat          GPE    548
                  kanada          GPE    527
                 meksiko          GPE    380
                   april          DAT    367
presiden amerika serikat          NOR    363
               indonesia          ORG    284
                presiden          NOR    279
                  persen          PRD    245
         

## 8. Export Results

In [8]:
# ── Flat CSV: one row per entity ────────────────────────────────────────────
# FLAT_CSV = "../outputs/ner_results_flat.csv"
# results_df.to_csv(FLAT_CSV, index=False)
# print(f"Flat results saved → {FLAT_CSV}  ({len(results_df):,} rows)")

# ── Per-article summary: entities grouped per article ───────────────────────
unique_df = results_df[~results_df["is_duplicate"]].copy()

unique_df["entity_record"] = unique_df.apply(
    lambda r: {"word": r["word"], "entity_group": r["entity_group"], "score": round(float(r["score"]), 4)},
    axis=1
)

summary_df = (
    unique_df
    .groupby(["article_idx", "judul", "url"], as_index=False, dropna=False)["entity_record"]
    .agg(list)
    .rename(columns={"entity_record": "entities"})
)

summary_df["entity_count"] = summary_df["entities"].apply(len)
summary_df["entities_json"] = summary_df["entities"].apply(json.dumps)

SUMMARY_CSV = "../outputs/ner_results_summary.csv"
summary_df.drop(columns=["entities"]).to_csv(SUMMARY_CSV, index=False)
print(f"Summary saved → {SUMMARY_CSV}  ({len(summary_df):,} articles)")

summary_df.drop(columns=["entities_json"]).head(5)

Summary saved → ../outputs/ner_results_summary.csv  (999 articles)


,article_idx,judul,url,entities,entity_count
0,0,Trump Sebar Exceutive Order: Emang Semengerika...,https://www.cnbcindonesia.com/research/2025012...,"[{'word': 'indonesia', 'entity_group': 'GPE', ...",31
1,1,"Alasan Rupiah 'Berpesta' di Pelantikan Trump, ...",https://www.cnbcindonesia.com/research/2025012...,"[{'word': 'indonesia', 'entity_group': 'GPE', ...",25
2,2,"Trump Beri Kabar Baik, Saatnya Menunggu Dolar ...",https://www.cnbcindonesia.com/research/2025012...,"[{'word': 'indonesia', 'entity_group': 'GPE', ...",31
3,3,"IHSG Merah Lagi, Begini Penjelasan dari Analis...",https://www.cnbcindonesia.com/research/2025030...,"[{'word': 'indonesia', 'entity_group': 'GPE', ...",54
4,4,Bernstein: Bitcoin Bisa Naik 2x Lipat! Target ...,https://indodax.com/academy/bitcoin-200k-predi...,"[{'word': 'btc', 'entity_group': 'PRD', 'score...",23
